# Healthier Recipe Optimization via Rule-Based Heuristics
**Author:** Kristian Hudák  
**Competition:** National Olympiad in Artificial Intelligence (OAI) — Home Qualifier Round

This notebook implements a rule-based expert heuristic system for recipe nutritional optimization:
- Computes full macronutrient profiles (calories, protein, carbohydrates, fat) from user inputs
- Queries the categorized USDA food database to find lower-calorie substitutes within the same food category
- Applies portion penalty reductions (e.g. 20% grammage reduction) when direct healthier substitutions are unavailable
- Interactive command-line recipe interface supporting fuzzy ingredient matching


In [13]:
# Heuristic function definitions for macronutrient calculation and database retrieval
def spocitaj_makra(zoznam_surovin, databaza):
    """
    Calculates kcal, protein, carbs, and fat for the whole recipe (list of tuples).
    """
    energia = bielkoviny = sacharidy = tuky = 0

    for nazov, gramy in zoznam_surovin:
        # Database lookup (using df_ingr reference table)
        riadok = databaza[databaza['description'] == nazov]

        if riadok.empty:
            continue  # Skip unrecognized ingredient

        data = riadok.iloc[0]
        pomer = gramy / 100.0

        energia     += data['calories'] * pomer
        bielkoviny  += data['protein'] * pomer
        sacharidy   += data['carbs'] * pomer
        tuky        += data['fat'] * pomer

    return energia, bielkoviny, sacharidy, tuky

def najdi_lepsiu_surovinu(nazov_suroviny, databaza):
    """
    Nájde zdravšiu alternatívu: rovnaká kategória, ale nižšia energia, tuky aj sacharidy.
    """
    riadok = databaza[databaza['description'] == nazov_suroviny]
    if riadok.empty:
        return nazov_suroviny # Vráti pôvodnú, ak sa nenájde

    data = riadok.iloc[0]
    kategoria = data['food_category_id']

    # Filtrovanie kandidátov podľa prísnych podmienok
    kandidati = databaza[
        (databaza['food_category_id'] == kategoria) &
        (databaza['calories'] < data['calories']) &
        (databaza['fat'] < data['fat']) &
        (databaza['carbs'] < data['carbs'])
    ]

    if kandidati.empty:
        return nazov_suroviny

    # Vyberieme možnosť s absolútne najnižšou energiou
    najlepsi_index = kandidati['calories'].idxmin()
    return kandidati.loc[najlepsi_index]['description']

print("✅ Heuristické funkcie načítané.")


✅ Heuristické funkcie načítané.


In [14]:
# Modification logic: ingredient substitution, grammage reduction, and protein preservation
def vytvor_alternativu(zoznam_surovin, databaza):
    """
    Generates a healthier recipe version: ingredient substitution or 20% grammage reduction.
    """
    nove_jedlo = []

    for nazov, gramy in zoznam_surovin:
        nova_surovina = najdi_lepsiu_surovinu(nazov, databaza)

        # If no substitute found -> apply 20% grammage reduction
        if nova_surovina == nazov:
            nove_gramy = gramy * 0.8
        else:
            nove_gramy = gramy

        nove_jedlo.append((nova_surovina, nove_gramy))

    return nove_jedlo

def skontroluj_pravidla(povodny_zoznam, novy_zoznam, databaza):
    """
    Overí splnenie 3 hlavných podmienok zadania (Kcal, Proteín, Sacharidy).
    """
    # Výpočet hodnôt
    e_orig, p_orig, s_orig, _ = spocitaj_makra(povodny_zoznam, databaza)
    e_alt, p_alt, s_alt, _    = spocitaj_makra(novy_zoznam, databaza)

    # 1. Podmienka: Kalórie musia klesnúť pod 75% pôvodných
    if e_alt > 0.75 * e_orig:
        return False
    # 2. Podmienka: Proteín nesmie klesnúť
    if p_alt <= p_orig:
        return False
    # 3. Podmienka: Sacharidy musia byť nižšie
    if s_alt >= s_orig:
        return False

    return True

def dopln_bielkoviny(povodny_zoznam, novy_zoznam, databaza):
    """
    Ak alternatíva nemá viac bielkovín, pridá vysokoproteínovú 'záchranu'.
    """
    _, p_orig, _, _ = spocitaj_makra(povodny_zoznam, databaza)
    _, p_alt, _, _  = spocitaj_makra(novy_zoznam, databaza)

    if p_alt > p_orig:
        return novy_zoznam

    # Hľadáme "čistý proteín" (viac ako 15g bielkovín, menej ako 200 kcal)
    zdroje_proteinu = databaza[
        (databaza['protein'] > 15) &
        (databaza['calories'] < 200)
    ]

    if zdroje_proteinu.empty:
        return novy_zoznam

    # Vyberieme ten s najvyšším obsahom bielkovín
    top_protein = zdroje_proteinu.loc[zdroje_proteinu['protein'].idxmax()]['description']

    # Pridáme 50g tejto suroviny
    novy_zoznam.append((top_protein, 50))
    return novy_zoznam

print("✅ Logika pravidiel pripravená.")


✅ Logika pravidiel pripravená.


In [10]:
import pandas as pd

# Load the food ingredients database
df = pd.read_excel('/content/pivoted_categorized_food_data.xlsx')

print("✅ Ingredients database (df_ingr) loaded successfully.")


✅ Databáza surovín (df_ingr) načítaná.


In [22]:
# # Interaktívna konzola
from thefuzz import process

print("="*75)
print("🥗 NUTRITIONAL OPTIMIZER: DETAILED SUBSTITUTION LOG 🥗")
print("="*75)
print("Enter ingredients (e.g., Shrimp 150, Butter 20). Type 'exit' to quit.")
print("-" * 75)

while True:
    vstup = input("\n📝 Enter recipe: ")

    if vstup.lower() in ['koniec', 'exit', 'stop']:
        print("Exiting... Bon appétit!")
        break

    polozky = [p.strip() for p in vstup.split(',')]
    moje_jedlo = []

    for p in polozky:
        casti = p.rsplit(' ', 1)
        if len(casti) > 1 and casti[1].replace('.', '', 1).isdigit():
            nazov_hladany = casti[0]
            gramy = float(casti[1])
        else:
            nazov_hladany = p
            gramy = 100.0

        najlepsia_zhoda = process.extractOne(nazov_hladany, df_ingr['description'])

        if najlepsia_zhoda and najlepsia_zhoda[1] > 60:
            presny_nazov = najlepsia_zhoda[0]
            moje_jedlo.append((presny_nazov, gramy))
        else:
            print(f"   ❌ Surovinu '{nazov_hladany}' sa nepodarilo v databáze nájsť.")

    if not moje_jedlo:
        continue

    # --- Výpočty ---
    kcal_o, prot_o, carb_o, fat_o = spocitaj_makra(moje_jedlo, df_ingr)

    # 1. Krok: Identifikácia zmien hneď pri generovaní (pred iteráciami gramáže)
    # Vytvoríme zoznam zmien porovnaním indexov
    alt_meal = vytvor_alternativu(moje_jedlo, df_ingr)

    # --- LOGOVANIE TRANSFORMÁCIE ---
    print("\n🔄 TRANSFORMÁCIA SUROVÍN:")
    for i in range(len(moje_jedlo)):
        p_ing, p_g = moje_jedlo[i]
        n_ing, _ = alt_meal[i]

        if p_ing == n_ing:
            print(f" • {p_ing}: Ponechané (nenájdená lepšia alternatíva v kategórii)")
        else:
            print(f" • {p_ing} ➡️ NAHRADENÉ ZA: {n_ing}")

    # 2. Krok: Iteratívne ladenie gramáže a proteínu
    alt_meal = dopln_bielkoviny(moje_jedlo, alt_meal, df_ingr)

    # Sledujeme, či bol pridaný extra proteínový zdroj
    povodny_pocet = len(moje_jedlo)

    splnene = False
    for i in range(5):
        if skontroluj_pravidla(moje_jedlo, alt_meal, df_ingr):
            splnene = True
            break
        alt_meal = [(ing, g * 0.85) for ing, g in alt_meal]
        alt_meal = dopln_bielkoviny(moje_jedlo, alt_meal, df_ingr)

    # Ak sa pridala nová surovina (proteín), vypíšeme ju
    if len(alt_meal) > povodny_pocet:
        extra_ing = alt_meal[-1]
        print(f" • PRIDANÝ EXTRA PROTEÍN: {extra_ing[0]} ({extra_ing[1]:.0f}g)")

    # --- Tabuľkový výstup ---
    kcal_a, prot_a, carb_a, fat_a = spocitaj_makra(alt_meal, df_ingr)

    print("\n📝 FINÁLNY RECEPT:")
    for ing, g in alt_meal:
        print(f"   - {ing}: {g:.1f}g")

    data_vysledok = [
        ["Pôvodné", f"{kcal_o:.1f}", f"{prot_o:.1f}g", f"{carb_o:.1f}g", f"{fat_o:.1f}g"],
        ["Alternatíva", f"{kcal_a:.1f}", f"{prot_a:.1f}g", f"{carb_a:.1f}g", f"{fat_a:.1f}g"]
    ]
    df_compare = pd.DataFrame(data_vysledok, columns=["Verzia", "Kalórie", "Proteín", "Sacharidy", "Tuky"])

    print("\n📊 NUTRIČNÉ POROVNANIE:")
    print(df_compare.to_string(index=False))

    if splnene:
        print("\n✨ STATUS: Všetky podmienky úspešne splnené.")
    else:
        print("\n⚠️ STATUS: Nepodarilo sa dosiahnuť 75% kalórií (skús ubrať pôvodné tuky).")
    print("-" * 75)


🥗 NUTRIČNÝ OPTIMALIZÁTOR: DETAILNÝ VÝPIS ZMIEN 🥗
Zadaj suroviny (napr.: Shrimp 150, Butter 20). Napíš 'koniec' pre ukončenie.
---------------------------------------------------------------------------

📝 Zadaj recept: Pork loin, Potatoes, Cream, Lard, Onion

🔄 TRANSFORMÁCIA SUROVÍN:
 • Restaurant, Latino, tamale, pork: Ponechané (nenájdená lepšia alternatíva v kategórii)
 • Potatoes, russet, without skin, raw ➡️ NAHRADENÉ ZA: Cucumber, with peel, raw
 • Peanut butter, creamy ➡️ NAHRADENÉ ZA: Soy milk, unsweetened, plain, shelf stable
 • Collards, raw ➡️ NAHRADENÉ ZA: Pickles, cucumber, dill or kosher dill
 • Onions, red, raw ➡️ NAHRADENÉ ZA: Lettuce, iceberg, raw
 • PRIDANÝ EXTRA PROTEÍN: Chicken, broiler or fryers, breast, skinless, boneless, meat only, cooked, braised (50g)

📝 FINÁLNY RECEPT:
   - Restaurant, Latino, tamale, pork: 68.0g
   - Cucumber, with peel, raw: 85.0g
   - Soy milk, unsweetened, plain, shelf stable: 85.0g
   - Pickles, cucumber, dill or kosher dill: 85.0g
   - 

KeyboardInterrupt: Interrupted by user

### Sample Test Recipes
You can test the system with arbitrary ingredients or the benchmark combinations:
- `Shrimp 150, Butter 20, Rice 100`
- `Pork loin raw 150, Sweet potatoes raw 200, Green beans raw 100, Sour cream 50`
- `Oatmeal raw 80, Milk whole 200, Banana raw 120, Peanut butter 30`
